In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt

# Leer CSV normal
df = pd.read_csv('../data/WorldClim_v2.csv')

# Convertir la columna geometry de texto a geometría (si está en WKT)
df['geometry'] = df['geometry'].apply(wkt.loads)

# Convertir a GeoDataFrame
df_geo = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')  # O el CRS que corresponda

# Revisar
print(df_geo.head())
print(df_geo.crs)

   Unnamed: 0 countryCode  elevation_DEM      genus       lat        lon  \
0           2          CO           2940  Espeletia  4.849999 -73.999999   
1           3          CO           2940  Espeletia  4.849999 -73.999999   
2           4          CO           2940  Espeletia  4.849999 -73.999999   
3           6          CO           3509  Espeletia  4.573999 -74.030999   
4           7          CO           3509  Espeletia  4.573999 -74.030999   

                 species      bio1       bio3      bio4  bio7   bio12  \
0    Espeletia corymbosa  1.175000  79.234962  2.938769  1.22  1119.0   
1     Espeletia argentea  1.175000  79.234962  2.938769  1.22  1119.0   
2  Espeletia grandiflora  1.175000  79.234962  2.938769  1.22  1119.0   
3    Espeletia corymbosa  0.832083  80.940598  3.222493  1.01  1169.0   
4  Espeletia grandiflora  0.832083  80.940598  3.222493  1.01  1169.0   

       bio15                                               .geo  \
0  33.725811  {"geodesic":false,"type

In [7]:
df.columns

Index(['Unnamed: 0', 'countryCode', 'elevation_DEM', 'genus', 'lat', 'lon',
       'species', 'bio1', 'bio3', 'bio4', 'bio7', 'bio12', 'bio15', '.geo',
       'geometry'],
      dtype='object')

In [3]:
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from pysal.model import spreg
from pysal.lib import weights
from pysal.explore import esda
from scipy import stats
import statsmodels.formula.api as sm
import numpy
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sbn

In [10]:
df[variable_names].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20220 entries, 0 to 20219
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   elevation_DEM  20220 non-null  int64  
 1   species        19279 non-null  object 
 2   bio1           20220 non-null  float64
 3   bio3           20220 non-null  float64
 4   bio4           20220 non-null  float64
 5   bio7           20220 non-null  float64
 6   bio12          20220 non-null  float64
 7   bio15          20220 non-null  float64
dtypes: float64(6), int64(1), object(1)
memory usage: 1.2+ MB


In [11]:
variable_names =['elevation_DEM', 'bio1', 'bio3', 'bio4', 'bio7', 'bio12', 'bio15']
knn = weights.KNN.from_dataframe(df, k=5)

In [12]:
import numpy as np

# Parámetros
rho = 0.4  # autocorrelación (0 = nada, 1 = máxima)
beta = np.random.uniform(-1, 1, size=len(variable_names))  # coeficientes aleatorios
epsilon = np.random.normal(0, 1, size=df.shape[0])  # ruido

# Variables climáticas
X = df[variable_names].values

# Matriz W normalizada (para SAR o simulación)
W = knn.full()[0]
W = W / W.sum(axis=1, keepdims=True)  # normalizar filas

# Generar Y con autocorrelación
# (I - rho * W)^-1 (X * beta + epsilon)
I = np.eye(W.shape[0])
A_inv = np.linalg.inv(I - rho * W)
Y_sim = A_inv @ (X @ beta + epsilon)

# Guardar Y en el dataframe
df['Y_sim'] = Y_sim

In [14]:
# escalar la variable Y_sim al rango de alturas deseado
# Por ejemplo, alturas entre 0.3 m y 3.0 m

min_height = 0.3
max_height = 3.0

# Escalar al rango deseado
Y_sim_scaled = (Y_sim - Y_sim.min()) / (Y_sim.max() - Y_sim.min())  # escala 0-1
Y_sim_scaled = Y_sim_scaled * (max_height - min_height) + min_height

df['Y_sim'] = Y_sim_scaled

In [16]:
df_geo['Y_sim'] = Y_sim_scaled

In [19]:
df_geo.to_file('../data/Data_Ysimule.geojson', driver='GeoJSON')